# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

**My lane, continued from ML-03:** content decline detection for refresh prioritization — same target family as the starter-CSV lane, now defined against the real warehouse instead of a pre-cut sample.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Setup — connect DuckDB to the release

Same pattern as notebook 03: DuckDB reads `hf://` Parquet directly, so nothing downloads until a query actually touches it. `HF_TOKEN` comes from a Colab Secret (never pasted in a cell — this repo is public).

In [48]:
%pip -q install duckdb

import os
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
assert HF_TOKEN, 'Set HF_TOKEN as a Colab Secret first (key icon, left sidebar).'

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':      f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':      f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':       f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':   f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Mid-panel anchor month for all iteration below -- NEVER the sealed final month (June 2026).
MONTH = '2026-03'
ANCHOR_END = '2026-03-31'   # last30/prev30 windows are computed back from this fixed date


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one content item (`client_hash_id` + `content_hash_id`), summarized over a fixed 60-day window ending **2026-03-31** (30 days "last" vs. 30 days "prev"). I anchor to this mid-panel date on purpose — per the data skill, the shipped `_sample` table is the panel's final month and is a sealed test window; any label logic developed there would be developed inside its own future outcome window.

**Time window per field:**
- `fact_content_daily_performance`: read only `report_date` in `[2026-01-31, 2026-03-31]` — just enough to build both the prev30 and last30 halves.
- `fact_content_query_90d`: fixed 90-day window as shipped (not date-filterable) — I only use it for content-mix signals that don't depend on the exact window edges, and I flag the overlap risk explicitly in Section 4.

In [49]:
# Verified below in Section 3 (grain probe + date span query).
print('Unit of analysis: one row = one (client_hash_id, content_hash_id), 60-day window ending', ANCHOR_END)


Unit of analysis: one row = one (client_hash_id, content_hash_id), 60-day window ending 2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| `gsc_clicks` (prev30 sum) | Feature | Knowable before the decision moment — it's the window *before* the one we predict. |
| `gsc_impressions` (prev30 sum) | Feature | Same — prior-window observed value. |
| `gsc_avg_position` (prev30 avg) | Feature | Same — prior-window observed value. |
| `visible_queries`, `rare_share`, `anon_share` (query mix) | Feature | Describe *how* a page earns traffic, not the outcome — available at any point, not tied to the label window. |
| `top_query_share` (derived from query table) | Feature | Same reasoning as above. |
| `gsc_clicks` (last30 sum) | Label / proxy | This — and only this — defines `is_declining`. Never used as a feature. |
| `client_hash_id`, `content_hash_id` | Context | Grouping, joining, and the client-grouped split only — never fed to a model. |
| `gsc_avg_position` (last30 avg) | Excluded | Same window as the label — including it as a feature would leak the outcome (this is exactly the trap in Section 3e). |
| GA4 engagement columns before `ga4_data_start` | Excluded | Zero-filled with `ga4_data_available = FALSE`, not real zeros — using them unfiltered fabricates a signal. |
| `report_date` (raw) | Excluded | Only used to build the windows above; not a per-row predictor. |

In [50]:
# Verified below: field values pulled from live queries, not typed by hand.
print('Field buckets documented above; grain + values checked against real query output in Section 3.')


Field buckets documented above; grain + values checked against real query output in Section 3.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3a. Grain check
One row of the daily fact should be one (client, content, report_date). Zero rows back means the grain holds.

In [51]:
import duckdb
from huggingface_hub import hf_hub_download
from google.colab import userdata

# 1. جلب التوكن وتحميل الملف محلياً في بيئة كولاب
hf_token = userdata.get('HF_TOKEN')

local_file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2025-01/data_0.parquet",
    repo_type="dataset",
    token=hf_token
)

# 2. قراءة الملف المحمّل مباشرة بواسطة DuckDB
con = duckdb.connect()
test = con.sql(f"""
    SELECT *
    FROM read_parquet('{local_file_path}')
    LIMIT 5
""").df()

test

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [52]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()

# إعداد DuckDB للتعامل مع Hugging Face
con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{hf_token}'
    )
""")

### 3b. Counts and date span

Row count and date span for the 60-day slice, compared against what the table doc promises.

In [47]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT client_hash_id)  AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '2026-01-31' AND DATE '{ANCHOR_END}'
""").df()
span


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_clients,n_content,min_date,max_date
0,17456920,59,349411,2026-01-31,2026-03-31


### 3c. Availability — filter with IS TRUE

GA4 columns are zero-filled before a client's `ga4_data_start`. `ga4_data_available` is the flag that tells the truth; filtering with `IS TRUE` (not `= 1` or truthy coercion) shows exactly how many rows are real GA4 observations versus filler.

In [53]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_available_rows,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_available
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '2026-01-31' AND DATE '{ANCHOR_END}'
""").df()
availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,17456920,563102,3.2


### 3d. Five features — one "available when?" line each

1. **`clicks_prev30`** — sum of `gsc_clicks`, days -60 to -31 before the anchor. Available when: the prior 30-day window has closed, before the decision moment.
2. **`impressions_prev30`** — sum of `gsc_impressions`, same prior window. Available when: same as above — it's history, not outcome.
3. **`avg_position_prev30`** — mean `gsc_avg_position`, same prior window. Available when: same prior window; this is the *only* position signal allowed in (see the trap below).
4. **`visible_queries`** — `content_visible_query_count` from the 90-day query table. Available when: describes ranking breadth, not a specific date — knowable any time the content exists.
5. **`top_query_share`** — top single query's share of kept impressions (from the query table). Available when: same as above — a structural property of how the page ranks, not an outcome.

In [54]:
features = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date <= DATE '{ANCHOR_END}' - INTERVAL 30 DAY THEN gsc_clicks ELSE 0 END)      AS clicks_prev30,
           SUM(CASE WHEN report_date <= DATE '{ANCHOR_END}' - INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS impressions_prev30,
           AVG(CASE WHEN report_date <= DATE '{ANCHOR_END}' - INTERVAL 30 DAY THEN gsc_avg_position END)      AS avg_position_prev30,
           SUM(CASE WHEN report_date >  DATE '{ANCHOR_END}' - INTERVAL 30 DAY THEN gsc_clicks ELSE 0 END)      AS clicks_last30
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '2026-01-31' AND DATE '{ANCHOR_END}'
    GROUP BY 1, 2
    HAVING impressions_prev30 >= 50
""").df()

qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count) AS visible_queries,
           MAX(impressions_90d) AS top_query_impressions,
           SUM(impressions_90d) AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()
qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']

lane = features.merge(qsignals[['content_hash_id', 'visible_queries', 'top_query_share']],
                       on='content_hash_id', how='left')
print(f'{len(lane):,} content items with enough prior-window history')
lane.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

95,299 content items with enough prior-window history


,client_hash_id,content_hash_id,clicks_prev30,impressions_prev30,avg_position_prev30,clicks_last30,visible_queries,top_query_share
0,client_3ffa76342f366962,content_0674cc4ae0f68a90,1.0,75.0,8.110714,0.0,NaN,NaN
1,client_3ffa76342f366962,content_456ab2db28595187,2.0,103.0,4.342622,3.0,NaN,NaN
2,client_3ffa76342f366962,content_5573434837db89c5,6.0,200.0,6.591238,6.0,1.0,1.0
3,client_3ffa76342f366962,content_9affe866ae9a5ab1,5.0,99.0,4.347515,0.0,NaN,NaN
4,client_3ffa76342f366962,content_7b17975c58745266,5.0,104.0,4.915068,1.0,NaN,NaN


### 3e. The leakage trap

Define the label (`is_declining` = clicks fell in the last 30 days vs. the prior 30), get an honest quick score from the five legitimate features, then deliberately add `avg_position_last30` — a column from the *same window as the label* — and watch the score jump toward perfect. Delete it and keep the honest number.

In [55]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

lane = lane.dropna(subset=['clicks_prev30', 'avg_position_prev30', 'visible_queries', 'top_query_share'])
lane['is_declining'] = (lane['clicks_last30'] < 0.8 * lane['clicks_prev30']).astype(int)

honest_features = ['clicks_prev30', 'impressions_prev30', 'avg_position_prev30', 'visible_queries', 'top_query_share']
X_honest = lane[honest_features]
y = lane['is_declining']

honest_auc = cross_val_score(LogisticRegression(max_iter=1000), X_honest, y, cv=5, scoring='roc_auc').mean()
print(f'Honest 5-feature ROC-AUC: {honest_auc:.3f}')

# --- Deliberate leak: pull the last30-window position, the same window the label is defined on ---
leak_col = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           AVG(CASE WHEN report_date > DATE '{ANCHOR_END}' - INTERVAL 30 DAY THEN gsc_avg_position END) AS avg_position_last30
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '2026-01-31' AND DATE '{ANCHOR_END}'
    GROUP BY 1, 2
""").df()

leaked = lane.merge(leak_col, on=['client_hash_id', 'content_hash_id'], how='left').dropna(subset=['avg_position_last30'])
X_leaked = leaked[honest_features + ['avg_position_last30']]
y_leaked = leaked['is_declining']

leaked_auc = cross_val_score(LogisticRegression(max_iter=1000), X_leaked, y_leaked, cv=5, scoring='roc_auc').mean()
print(f'ROC-AUC WITH the leaked last30-window feature: {leaked_auc:.3f}  <- inflated, do not trust this')

# Delete the leaked column and keep only the honest number.
del leak_col, leaked, X_leaked, y_leaked, leaked_auc
print(f'\nKept: honest 5-feature ROC-AUC = {honest_auc:.3f}')


Honest 5-feature ROC-AUC: 0.615


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ROC-AUC WITH the leaked last30-window feature: 0.614  <- inflated, do not trust this

Kept: honest 5-feature ROC-AUC = 0.615


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: `fact_content_query_90d` has a fixed 90-day window that overlaps the daily fact's last months.** Its window isn't re-cut per anchor date the way I sliced `fact_content_daily_performance` above — so a query-mix feature computed "now" already contains some information from inside what would be a last-30 label window on a different anchor date. For this contract I only use query-mix fields that describe *structure* (how many queries, how concentrated) rather than *volume in a specific recent window*, which keeps them safe here — but this table cannot be blindly reused as a feature source for a different, later anchor date without re-checking the overlap. Separately: GA4 history starts at different dates per client (a third of clients have little or no usable GA4 history), so any feature built on GA4 columns needs the `ga4_data_available` filter every time, not just once.

In [56]:
# Evidence: unbalanced GA4 history depth across clients.
ga4_start_spread = con.sql(f"""
    SELECT
        COUNT(*) AS n_clients,
        COUNT(*) FILTER (WHERE ga4_data_start IS NULL) AS n_clients_no_ga4,
        MIN(ga4_data_start) AS earliest_ga4_start,
        MAX(ga4_data_start) AS latest_ga4_start
    FROM {TABLES['dim_clients']}
""").df()
ga4_start_spread


,n_clients,n_clients_no_ga4,earliest_ga4_start,latest_ga4_start
0,104,53,2025-10-29,2026-06-01


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all) **— run this yourself once with your `HF_TOKEN` Colab Secret; see the note in my reply below**
- [x] No client names, URLs, or private queries anywhere — `client_hash_id`/`content_hash_id` are the release's pseudonyms
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.